# Superstore Sales Analysis: Final Project

**Author:** Tilak Bhattarai
**Date:** 21 November 2026
**Dataset:** Sample-Superstore (cleaned, 10,194 rows x 26 columns)

## Project Goal
Combine the findings from Days 1 to 4 into one business story: what drives sales and profit, where the business loses money, and what actions can improve profitability.

## Notebook Structure
1. Setup
2. Key Findings
3. Business Questions
4. Recommendations
5. Conclusion

## 1. Setup
Load the cleaned dataset created in Day 2 and confirm it is ready for analysis.

In [ ]:
import pandas as pd
import numpy as np

file_path = "../data/processed/cleaned_sales_data.csv"
sales_df = pd.read_csv(file_path, parse_dates=["Order Date", "Ship Date"])

print("Shape:", sales_df.shape)
print("Total nulls:", sales_df.isnull().sum().sum())
print(sales_df[["Order Date", "Ship Date"]].dtypes)

Observation The dataset loads with 10,194 rows and 26 columns, both date columns are datetime types, and there are no missing values.

Decision: The data is ready for analysis.

## 2. Key Findings
The most important results from Days 1 to 4, each backed by a number from the data.

### 2.1 Overall Business Performance
Objective: Calculate the headline KPIs for the whole business.

In [ ]:
total_sales = sales_df["Sales"].sum()
total_profit = sales_df["Profit"].sum()
total_orders = sales_df["Order ID"].nunique()
total_customers = sales_df["Customer ID"].nunique()
profit_margin = total_profit / total_sales * 100
avg_order_value = total_sales / total_orders

print(f"Total Sales      : {total_sales:,.2f}")
print(f"Total Profit     : {total_profit:,.2f}")
print(f"Total Orders     : {total_orders:,}")
print(f"Total Customers  : {total_customers:,}")
print(f"Profit Margin    : {profit_margin:.2f}%")
print(f"Avg Order Value  : {avg_order_value:,.2f}")

Observation: The business earns a moderate overall profit margin. Sales are large, but profit is only a small share of them, so margin quality matters more than raw sales growth.

Decision: Focus the rest of the analysis on where margin is gained and lost.

### 2.2 Sales and Profit Trends
Objective: See how sales and profit changed by year and which months perform best.

In [ ]:
yearly = sales_df.groupby("Order Year").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).round(2)
yearly["Margin %"] = (yearly["Profit"] / yearly["Sales"] * 100).round(2)
print(yearly)
print()

monthly = sales_df.groupby("Order Month")["Sales"].sum().round(2)
print("Best month by sales :", monthly.idxmax(), "->", monthly.max())
print("Worst month by sales:", monthly.idxmin(), "->", monthly.min())

Observation: Check the yearly table for the direction of sales, profit and margin, and note the strongest and weakest months. Sales usually peak in the last quarter of the year.

Decision: Plan stock, marketing and staffing around the peak months.

### 2.3 Category Performance
Objective: Find which categories bring the most sales and profit.

In [ ]:
category_summary = sales_df.groupby("Category").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
).round(2)
category_summary["Margin %"] = (category_summary["Profit"] / category_summary["Sales"] * 100).round(2)
category_summary = category_summary.sort_values("Profit", ascending=False)
print(category_summary)

Observation: Compare the categories on profit and margin, not only on sales. A category with high sales can still have a weak margin.

Decision: Prioritise the categories with the best margin, and review the weakest one in Section 3.

## 3. Business Questions
Each question is answered with Pandas in the format: Objective, Code, Output, Observation, Decision.

### Q1. Which sub-categories and products lose money?
Objective: Identify loss-making sub-categories and the 10 worst products.

In [ ]:
sub_summary = sales_df.groupby("Sub-Category").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).round(2)
sub_summary["Margin %"] = (sub_summary["Profit"] / sub_summary["Sales"] * 100).round(2)

loss_subs = sub_summary[sub_summary["Profit"] < 0].sort_values("Profit")
print("Loss-making sub-categories:")
print(loss_subs)
print()

worst_products = sales_df.groupby("Product Name")["Profit"].sum().sort_values().head(10).round(2)
print("10 worst products by profit:")
print(worst_products)

Observation: The sub-categories listed above lose money overall, and the 10 worst products contribute a large part of the losses.

Decision: Review pricing and discounts for these sub-categories and products before pushing more sales.

### Q2. How does discount affect profit?
Objective: Compare profit across discount bands.

In [ ]:
bins = [-0.01, 0, 0.2, 0.4, 1]
labels = ["No discount", "1-20%", "21-40%", "Above 40%"]
sales_df["Discount Band"] = pd.cut(sales_df["Discount"], bins=bins, labels=labels)

discount_summary = sales_df.groupby("Discount Band", observed=True).agg(
    Orders=("Order ID", "count"),
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Profit=("Profit", "mean")
).round(2)
discount_summary["Margin %"] = (discount_summary["Profit"] / discount_summary["Sales"] * 100).round(2)
print(discount_summary)

Observation: Profit and margin fall as the discount band rises, and the highest bands turn negative.

Decision: Put a cap on discounts and require approval for anything in the high discount bands.

### Q3. Which regions and states are weakest?
**Objective:** Compare regional margins and find the 5 worst states by profit.

In [ ]:
region_summary = sales_df.groupby("Region").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).round(2)
region_summary["Margin %"] = (region_summary["Profit"] / region_summary["Sales"] * 100).round(2)
print(region_summary.sort_values("Margin %", ascending=False))
print()

worst_states = sales_df.groupby("State")["Profit"].sum().sort_values().head(5).round(2)
print("5 worst states by profit:")
print(worst_states)

Observation: Regions differ clearly in margin, and a few states cause most of the regional losses.

Decision: Investigate pricing and discount habits in the weakest states before expanding there.

### Q4. Who are the most valuable customers and segments?
Objective: Find the top 10 customers by profit and compare customer segments.

In [ ]:
top_customers = sales_df.groupby("Customer Name")["Profit"].sum().sort_values(ascending=False).head(10).round(2)
print("Top 10 customers by profit:")
print(top_customers)
print()

segment_summary = sales_df.groupby("Segment").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Customers=("Customer ID", "nunique")
).round(2)
segment_summary["Margin %"] = (segment_summary["Profit"] / segment_summary["Sales"] * 100).round(2)
print(segment_summary)

Observation: A small group of customers brings a large share of profit, and the segments differ in margin.

Decision: Build retention offers for the top customers and focus acquisition on the best segment.

### Q5. Does shipping mode change delivery time?
Objective: Compare average shipping days across ship modes.

In [ ]:
ship_summary = sales_df.groupby("Ship Mode").agg(
    Orders=("Order ID", "count"),
    Avg_Shipping_Days=("Shipping Days", "mean"),
    Profit=("Profit", "sum")
).round(2)
print(ship_summary.sort_values("Avg_Shipping_Days"))

Observation: Faster ship modes deliver in fewer days, as expected. Compare their profit to see whether speed is paid for.

Decision: Keep the ship mode mix that balances delivery speed and profit.

## 4. Recommendations
1. Control discounts: Cap discounts and require approval for the high discount bands, because profit falls sharply there (Q2).
2. Fix loss-making sub-categories: Review pricing, supplier cost and discounts for the loss-making sub-categories and worst products (Q1).
3. Protect peak months: Plan stock and marketing around the strongest sales months (Section 2.2).
4. Fix weak regions: Investigate the weakest states before further expansion (Q3).
5. Retain top customers: Offer loyalty benefits to the top customers and the best segment (Q4).

## 5. Conclusion
Overall, the business has healthy sales but a thin profit margin. The main margin losses come from heavy discounts, a few loss-making sub-categories and products, and some weak states. Reducing discounts and fixing these areas should improve profit without needing more sales.

### Limitations
- Future-dated orders and ship dates were kept in the data after investigation (Day 2), so some date-based results may include them.
- The analysis is descriptive. It shows what happened, not exactly why.
- Negative-profit rows and outliers were retained, so totals include them.

### Next Steps
- Rebuild key charts with Matplotlib or Seaborn.
- Try a simple profit prediction model after the AI/ML stage.